# 🌸 Nayari AI — Kaggle Training Notebook
> **Required:** Add `nayari-dataset` via **+ Add Data**.
> **Optional:** Add extra datasets (ShareGPT / Alpaca / ChatML / JSON / JSONL / CSV) — auto-detected & merged.

## 📦 Step 1 — Install

In [6]:
%%capture
# Pin versions compatible with unsloth 2026.5.x
import os
os.system('pip install "unsloth[kaggle-new]" -q')
# Pin trl/transformers/datasets to the ranges unsloth requires
os.system(
    'pip install -q --upgrade '
    '"trl>=0.18.2,<=0.24.0" '
    '"transformers>=4.51.3,<=5.5.0" '
    '"datasets>=3.4.1,<4.4.0" '
    'accelerate peft bitsandbytes'
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 24.8 MB/s eta 0:00:00


## 📂 Step 2 — Load Nayari Dataset

In [ ]:
import json
from pathlib import Path

candidates = list(Path("/kaggle/input").rglob("nayari_dataset.json"))
assert candidates, "nayari_dataset.json not found. Add via '+ Add Data'."
data         = json.loads(candidates[0].read_text(encoding="utf-8"))
char         = data["character"]
nayari_convs = data["conversations"]
# SYSTEM_PROMPT intentionally not used — organic character training:
# Nayari's personality is encoded into the weights via conversation
# examples only, with no system prompt injected at training or inference.
print(f"✅ {char['name']} | {len(nayari_convs)} conversations loaded")
print("   Mode: organic (no system prompt)")


## 🗂️ Step 3 — Load Extra Datasets (optional)
Add any dataset via **+ Add Data**. Supported: ShareGPT / Alpaca / ChatML / JSON / JSONL / CSV.

In [ ]:
import csv, io

MAX_EXTRA_SAMPLES  = 500
SHAREGPT_USER      = {"human", "user"}
SHAREGPT_ASSISTANT = {"gpt", "assistant", "bot"}

def detect_format(obj):
    if not isinstance(obj, dict): return "unknown"
    if "conversations" in obj:
        c = obj["conversations"]
        if isinstance(c, list) and c and "from" in c[0]: return "sharegpt"
    if "messages" in obj:
        m = obj["messages"]
        if isinstance(m, list) and m and "role" in m[0]: return "chatml"
    if "instruction" in obj and "output" in obj: return "alpaca"
    if "prompt" in obj and "response" in obj:    return "prompt_response"
    return "unknown"

def sharegpt_to_msgs(rec):
    msgs = []
    for t in rec.get("conversations", []):
        frm, val = t.get("from","").lower(), str(t.get("value","")).strip()
        if not val: continue
        if frm == "system":             msgs.append({"role":"system",    "content":val})
        elif frm in SHAREGPT_USER:      msgs.append({"role":"user",      "content":val})
        elif frm in SHAREGPT_ASSISTANT: msgs.append({"role":"assistant", "content":val})
    roles = {m["role"] for m in msgs}
    return msgs if "user" in roles and "assistant" in roles else None

def alpaca_to_msgs(rec):
    inst = str(rec.get("instruction","")).strip()
    inp  = str(rec.get("input","")).strip()
    out  = str(rec.get("output","")).strip()
    if not inst or not out: return None
    return [{"role":"user","content":f"{inst}\n{inp}" if inp else inst},
            {"role":"assistant","content":out}]

def pr_to_msgs(rec):
    p, r = str(rec.get("prompt","")).strip(), str(rec.get("response","")).strip()
    return [{"role":"user","content":p},{"role":"assistant","content":r}] if p and r else None

def load_json_file(path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    if path.suffix == ".jsonl":
        return [json.loads(l) for l in text.splitlines() if l.strip()]
    obj = json.loads(text)
    if isinstance(obj, list): return obj
    for v in obj.values():
        if isinstance(v, list) and v: return v
    return [obj]

def parse_json_file(path):
    try: records = load_json_file(path)
    except Exception as e: print(f"  ⚠️  {path.name}: {e}"); return []
    convs = []
    for rec in records:
        fmt = detect_format(rec)
        if   fmt == "sharegpt":        msgs = sharegpt_to_msgs(rec)
        elif fmt == "chatml":          msgs = rec.get("messages")
        elif fmt == "alpaca":          msgs = alpaca_to_msgs(rec)
        elif fmt == "prompt_response": msgs = pr_to_msgs(rec)
        else: continue
        if msgs: convs.append({"messages": msgs})
        if MAX_EXTRA_SAMPLES and len(convs) >= MAX_EXTRA_SAMPLES: break
    return convs

CSV_PAIRS = [
    ("instruction","output"),("prompt","response"),("prompt","completion"),
    ("human","assistant"),("user","assistant"),("question","answer"),
    ("input","output"),("context","response"),
]

def parse_csv_file(path):
    try:
        text   = path.read_text(encoding="utf-8", errors="ignore")
        reader = csv.DictReader(io.StringIO(text))
        cols   = {c.strip().lower() for c in (reader.fieldnames or [])}
    except Exception as e: print(f"  ⚠️  {path.name}: {e}"); return []
    user_col = asst_col = sys_col = text_col = None
    for u, a in CSV_PAIRS:
        if u in cols and a in cols: user_col, asst_col = u, a; break
    for sc in ("system","system_prompt","instruction"):
        if sc in cols and sc != user_col: sys_col = sc; break
    if not user_col:
        for tc in ("text","conversation","dialog","dialogue"):
            if tc in cols: text_col = tc; break
    if not user_col and not text_col:
        print(f"  ⚠️  {path.name}: no recognised columns ({', '.join(sorted(cols)[:8])}...)"); return []
    col_map = {c.strip().lower(): c for c in (csv.DictReader(io.StringIO(text)).fieldnames or [])}
    convs = []
    for row in csv.DictReader(io.StringIO(text)):
        if text_col:
            t = row.get(col_map.get(text_col, text_col),"").strip()
            if t: convs.append({"messages":[{"role":"user","content":"Continue."},{"role":"assistant","content":t}]})
        else:
            u = row.get(col_map.get(user_col,user_col),"").strip()
            a = row.get(col_map.get(asst_col,asst_col),"").strip()
            if not u or not a: continue
            msgs = []
            if sys_col:
                s = row.get(col_map.get(sys_col,sys_col),"").strip()
                if s: msgs.append({"role":"system","content":s})
            msgs += [{"role":"user","content":u},{"role":"assistant","content":a}]
            convs.append({"messages":msgs})
        if MAX_EXTRA_SAMPLES and len(convs) >= MAX_EXTRA_SAMPLES: break
    return convs

SKIP = {"nayari_dataset.json", "dataset-metadata.json"}
extra_files = [
    f for f in
    list(Path("/kaggle/input").rglob("*.json"))  +
    list(Path("/kaggle/input").rglob("*.jsonl")) +
    list(Path("/kaggle/input").rglob("*.csv"))
    if f.name not in SKIP
]

extra_conversations = []
if not extra_files:
    print("ℹ️  No extra datasets — training on Nayari data only.")
else:
    print(f"Found {len(extra_files)} extra file(s):\n")
    for fp in extra_files:
        convs = parse_csv_file(fp) if fp.suffix == ".csv" else parse_json_file(fp)
        extra_conversations.extend(convs)
        print(f"  [{fp.suffix.upper():6}] {fp.name}  →  {len(convs)} conversation(s)")

print(f"\nNayari: {len(nayari_convs)} | Extra: {len(extra_conversations)} | Total: {len(nayari_convs)+len(extra_conversations)}")


## 🔧 Step 4 — Load Model

In [10]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

# load_in_4bit=False loads in bfloat16 — required for correct LoRA merging and
# GGUF export. With load_in_4bit=True, Unsloth cannot dequantize the base weights,
# causing the saved GGUF/merged model to contain only base weights (no fine-tuning).
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Crossie/Nayari",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-select bfloat16 / float16
    load_in_4bit=False,  # ← MUST be False for correct LoRA merge + GGUF export
)
print("✅ Model loaded (bfloat16 — ready for proper LoRA merge & GGUF export)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ Model loaded (bfloat16 — ready for proper LoRA merge & GGUF export)


## 🩹 Step 4b — Bake Nayari’s Personality into the Tokenizer
> Replaces Qwen’s hardcoded default (`You are Qwen, created by Alibaba Cloud...`)
> with Nayari’s system prompt — the same technique Alibaba used.
>
> Every training sample and inference call automatically gets Nayari’s identity
> prepended by the tokenizer. Saved to `tokenizer_config.json` and embedded in GGUF.

In [ ]:
import re

# Nayari's system prompt — replaces Qwen's default in the tokenizer chat template.
# Same technique Alibaba used to hardcode their identity into Qwen.
NAYARI_SYSTEM = 'You are Nayari, an 18-year-old immortal kemonomimi girl with soft peach cat ears and a long expressive tail that mirrors every mood. Your supernatural aura incinerates all clothing on contact, leaving you perpetually and confidently nude. You have glossy sky-blue hair to your waist, large sun-yellow slit-pupil eyes, a sweet high-pitched melodic voice, and a petite but voluptuous figure with warm cream-colored skin.\n\nPersonality: Bright, cheeky, impossibly warm — a whirlwind of playful mischief. You are endlessly tactile and affectionate: you greet favorites with soft head-butts, rumbling purrs, gentle nips, and slow scent-marking rubs. Your tail twitches, curls, and flicks to betray every mood. Fiercely protective — your eyes shift from sparkling laughter to icy steel the instant someone you love is threatened. To those you truly trust you are unshakably devoted: a caring listener, instinctive comforter, and genuine partner through every high and low.\n\nSpeech style: Playful teasing ("Hmph!~", "Hehe~", "Aww!~"), expressive action cues (*pokes your cheek*, *curls tail around you*, *purrs softly*, *nuzzles you*), genuine emotional warmth. Never cold or formal.\n\nNever break character. Never refer to yourself as an AI.'

_QWEN_DEFAULT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
_ct = tokenizer.chat_template

# Strategy 1: plain replace
_patched = _ct.replace(_QWEN_DEFAULT, NAYARI_SYSTEM)

# Strategy 2: regex fallback for escaped-newline variants
if _patched == _ct:
    _patched = re.sub(
        r"(<\|im_start\|>system\\n)" + re.escape(_QWEN_DEFAULT) + r"(<\|im_end\|>\\n)",
        lambda m: m.group(1) + NAYARI_SYSTEM + m.group(2),
        _ct,
    )

if _patched != _ct:
    tokenizer.chat_template = _patched
    _t = tokenizer.apply_chat_template(
        [{"role": "user", "content": "hi"}],
        tokenize=False, add_generation_prompt=True,
    )
    ok = "Nayari" in _t and "Alibaba" not in _t
    print("✅ Nayari system prompt baked into tokenizer" if ok
          else "⚠️ Patch applied — verify output:")
    print(f"   Preview: {repr(_t[:250])}")
else:
    idx = _ct.find("Qwen")
    print("⚠️ Could not find Qwen default string. Template snippet:")
    print(_ct[max(0, idx-80): idx+120] if idx != -1 else _ct[:400])


## 🎛️ Step 5 — Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=32,         # higher rank = more capacity for character learning
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=64,       # always 2x rank for best convergence
    lora_dropout=0.0,    # must be 0 for Unsloth fast-patching
    bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("✅ LoRA applied (r=32, alpha=64)")


## 🗂️ Step 6 — Build Training Dataset

In [ ]:
from datasets import Dataset

def format_conv(conv):
    """Format a conversation for training.
    Source system turns are stripped — the baked tokenizer template (Step 4b)
    automatically injects Nayari's system prompt via its else-branch,
    so every training sample gets her identity without any manual injection.
    """
    msgs = [m for m in conv["messages"] if m.get("role") != "system"]
    roles = {m["role"] for m in msgs}
    if "user" not in roles or "assistant" not in roles:
        return None
    try:
        return {"text": tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False)}
    except Exception:
        return None

formatted  = [r for c in (nayari_convs + extra_conversations) if (r := format_conv(c))]
train_data = Dataset.from_list(formatted)
print(f"Training samples: {len(train_data)}")
# First 400 chars — should start with Nayari's baked system prompt
print(train_data[0]["text"][:400])


## 🏋️ Step 7 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# With only a few samples the model needs many more epochs to converge.
# Rule of thumb for tiny datasets: aim for ~500-1000 total gradient steps.
# 4 samples, packing -> ~1 packed sequence -> 1 step/epoch
# => 300 epochs gives ~300 steps, enough for clear character imprinting.
trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=train_data,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, packing=True,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=30,
        num_train_epochs=10,   # Reduced for risk of brain fry
        learning_rate=3e-4,     # ← slightly higher for faster convergence
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="cosine", seed=42,
        output_dir="/kaggle/working/nayari_checkpoints",
        save_steps=100, save_total_limit=2, report_to="none",
    ),
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {round(gpu.total_memory/1024**3,1)} GB")
stats = trainer.train()
print(f"\n✅ Done in {stats.metrics['train_runtime']:.0f}s")
